# 🔥 guato_ba — Acumulado anual (o produto final)
### De um par de datas ao mapa de queimadas do Pantanal inteiro
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

> ⚠️ **Notebook descritivo.** As células de código abaixo mostram *como* o produto é gerado, mas **não são executadas aqui** porque precisam de acesso ao **Brazil Data Cube** (rede) e de **GPU**. Para rodar de verdade, use a CLI indicada em cada passo. Os produtos prontos ficam em `results/acumulado/`.

Um par de datas detecta a queimada **daquele intervalo**. O **produto anual** cobre o ano inteiro: roda o modelo em **todos os pares consecutivos** de datas limpas e **une** as detecções — tudo **tile-a-tile** (sem mosaico global gigante), o que o torna **retomável**.

</div>

# <span style="color:#cf4c39">🔁 1. O pipeline tile-a-tile</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Para cada tile MGRS do Pantanal, de janeiro a dezembro:

1. seleciona datas com **nuvem < 10%**;
2. compõe as revisitas do dia e mantém só datas com **cobertura válida ≥ 50%** do tile;
3. forma **pares consecutivos** (T1→T2) e aplica o modelo **só nos pixels válidos** (SCL+WIW);
4. **une** (OR lógico) todas as detecções do ano;
5. **recorta ao bioma** (guardrail) e remove manchas menores que a **MMU de 24 px (~1 ha)**;
6. reporta a **% do Pantanal efetivamente analisada** (para comparação justa com HLS/DETER/MapBiomas).

</div>

In [ ]:
# ILUSTRATIVO — não executar aqui (precisa de BDC + GPU)
import os
os.environ["GUATO_NORM"] = "gn"              # casar com o checkpoint treinado
import accumulate_tile as A

A.INI, A.FIM = "2024-01-01", "2024-12-31"
resumo = A.run(
    modelo="biatt_unet",
    tiles=["21KWB", "21KVB"],
    ckpt="deeplearning/biatt_unet/biatt_unet_best.pkl",
    thr=0.90,                                # limiar de probabilidade (calibrado)
    tag="_demo",
)
resumo   # dict com área queimada (ha) e % analisada por tile

# <span style="color:#cf4c39">🖥️ 2. Rodando pela linha de comando</span>
<hr style="border:1px solid #e69627;">

```bash
P=/opt/conda/envs/geospatial/bin/python

# um ou poucos tiles
$P main.py accumulate      --modelo biatt_unet --cloud 10 --in_bioma 0.50 \
                           --thr 0.90 --mmu 24 --tiles 21KWB,21KVB

# Pantanal inteiro (grava um checkpoint .npz por tile -> retomável)
$P main.py accumulate-full --modelo biatt_unet --cloud 10 --in_bioma 0.50 --thr 0.90 --mmu 24

# acumulado por satélite Landsat (transfer da DL de 3 bandas)
$P accumulate_landsat.py   biatt_unet
```

Cada execução grava um **raster** (`.tif`) e um **vetor** (`.gpkg`) com a **data de detecção por polígono**, em `results/acumulado/`. Como escreve um `.npz` por tile, reiniciar **reaproveita** os tiles já prontos.

# <span style="color:#cf4c39">📦 3. O produto do estudo</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

O produto de referência do estudo usa a **`biatt_unet`** com limiar **THR=0,90** e **MMU=24 px**, combinando Sentinel-2 e Landsat para maximizar a cobertura. As tabelas finais de área e de concordância com HLS/DETER/MapBiomas estão em `results/artigo/` (ver notebook 04) e os logs de execução em `logs/`.

</div>

In [ ]:
# conferir os produtos já gerados neste checkout
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT/"results").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
acum = ROOT/"results/acumulado"
if acum.exists():
    for p in sorted(acum.glob("acum*.*"))[:12]:
        print(p.name)
else:
    print("results/acumulado/ não está neste checkout (dados pesados ficam fora do git — ver DATA.md)")

---
Para um passeio **reproduzível** de ponta a ponta (BDC → cicatriz), veja **`06_tutorial_ponta_a_ponta`**.